# AmbiQT Three-Agent Colab Pilot (v2, hardened orchestration)

Runs Agent A, A2, B on the same 12 records with Qwen3:4B via local Ollama and
scores with the real Stage 2 pipeline.

**Run the cells in order.** Cell 5 is a server gate: do not run the pilot (Cell 7)
unless the smoke test passes. Agent prompts, generation config, scoring, golds,
benchmark data, and the 12-record selection are unchanged from v1.

## Cell 1 — upload & unzip

In [ ]:
from google.colab import files
up = files.upload()                       # choose ambiqt_colab_pilot.zip
!unzip -q ambiqt_colab_pilot.zip -d /content
%cd /content/colab_pilot
!ls

## Cell 2 — Python requirements

In [ ]:
!pip -q install -r requirements.txt       # sqlglot==30.31.0

## Cell 3 — verify the bundle (NO model inference)
Expect `RESULT: ALL CHECKS PASS`.

In [ ]:
!python prepare_and_verify.py

## Cell 4 — install Ollama, start server, pull Qwen3:4B
No fixed sleep gate here on purpose — Cell 5 polls for readiness.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess
subprocess.Popen(["ollama", "serve"])     # background server
!ollama pull qwen3:4b                      # blocks until model present

## Cell 5 — SERVER GATE: readiness + exact-tag model check + 1-request smoke test
Expect `RESULT: SMOKE TEST PASS`. If it fails, re-run Cell 4, wait, and re-run this.

In [ ]:
!python smoke_test.py

## Cell 6 — mount Google Drive for persistent results

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/ambiqt_pilot'
os.makedirs(OUT, exist_ok=True)

## Cell 7 — run the live pilot (12 × 3 = 36 generations)
Has its own preflight (exit 2 if server unreachable) and fail-fast (exit 3 if >20% request_error, partial results preserved). Clean run ends `RUN OK: 36/36 ...`.

In [ ]:
!python run_pilot.py --out "/content/drive/MyDrive/ambiqt_pilot"

## Cell 8 — quick look at budget / health

In [ ]:
import json, collections
run = sorted([f for f in os.listdir(OUT) if f.startswith("pilot_generations_")])[-1]
rows = [json.loads(l) for l in open(os.path.join(OUT, run))]
by = collections.defaultdict(list)
for r in rows: by[r["agent"]].append(r)
for a, rs in by.items():
    tok = [r["total_generated_tokens"] or 0 for r in rs]
    trunc = sum(1 for r in rs if r["done_reason"] == "length")
    perr  = sum(1 for r in rs if r["generation_status"] == "parse_error")
    rerr  = sum(1 for r in rs if r["generation_status"] == "request_error")
    print(f"{a}: req_err={rerr}/{len(rs)} trunc(len)={trunc}/{len(rs)} parse_err={perr} "
          f"max_tok={max(tok)} mean_lat={sum(r['latency'] for r in rs)/len(rs):.1f}s")

## (optional) orchestration unit tests — no live Ollama
Expect `RESULT: ALL ORCHESTRATION TESTS PASS`.

In [ ]:
!python test_orchestration.py